In [1]:
# Environment Setting
import cx_Oracle
from datetime import datetime, timedelta
import holidays
from IPython.display import display
from ipywidgets import GridspecLayout, VBox, Layout, widgets
import itertools
import logging
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import seaborn as sns
from scipy.stats.mstats import winsorize
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from tabulate import tabulate
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tqdm import tqdm
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Dataframe Setting & Cleaning - Activity Detail
Revenues       = pd.read_excel(r"data.xlsx")
Revenues['ds'] = pd.to_datetime(Revenues['ds'])
Revenues       = Revenues[Revenues['ds'] <= pd.Timestamp.today()]

TeamMap = {'ACCOUNT MANAGEMENT': 'Commercial',
           'ADMINISTRATION': 'Corporate', 
           'ASC-DE': 'Commercial', 
           'BLOOM': 'Workplace',
           'BRAZIL': 'Commercial', 
           'BUILDING MAINTENANCE MGMT': 'Engineering', 
           'CEO': 'Corporate',
           'CHILE': 'Commercial', 
           'CONTACT CENTER': 'OperationCenter', 
           'DIGITAL MARKETING & COMM': 'Corporate', 
           'DIGITAL PRODUCT DESIGN': 'Design',
           'DUE DILIGENCE': 'DueDiligence', 
           'EFM4RETAIL': 'Retail', 
           'ENERGY & SUSTAINABILITY': 'Energy', 
           'ENGINEERING': 'Engineering', 
           'ESSE-RE': 'EsseRe', 
           'FACILITY DELIVERY': 'OperationCenter', 
           'FACILITY DEMAND': 'FacilityPlatform', 
           'FACILITY PLATFORM': 'FacilityPlatform', 
           'FINANCE': 'Corporate', 
           'FMP': 'Commercial', 
           'GENERAL TEAM': 'Corporate', 
           'GERMANY': 'Commercial',
           'ICT': 'ICT', 
           'ICT - SERVICE': 'ICT',
           'ITALY': 'Commercial', 
           'LATAM': 'Commercial',
           'MARKET': 'Commercial', 
           'MITEU': 'Commercial', 
           'NA': 'Other', 
           'None': 'Other', 
           'NORAM': 'Commercial', 
           'OPERATION CENTER': 'OperationCenter', 
           'PARENT HUG': 'Workplace',
           'PEOPLE, PLACE &COMPLIANCE': 'Corporate', 
           'PLACE EVOLUTION': 'PlaceEvolution', 
           'PPP': 'OperationCenter', 
           'PROJECT': 'Project', 
           'PROPERTY': 'Property', 
           'RE-CON': 'ReCon', 
           'REENGINEERING - FACILITY': 'OperationCenter', 
           'REENGINEERING - PROJECT': 'Project', 
           'REENGINEERING - PROPERTY': 'Property', 
           'SDP AMS': 'TechCareServices', 
           'SERVICE DESIGN': 'ServiceDesign', 
           'SPACE MANAGEMENT': 'SpaceManagement', 
           'TECH CARE SERVICES': 'TechCareServices', 
           'TECHNOLOGY SERVICES': 'TechnologyServices', 
           'TURKEY': 'Commercial',
           'UNITED ARAB EMIRATES': 'Commercial',
           'WORKPLACE': 'Workplace'}

Revenues['t']  = Revenues['t'].str.upper().replace(TeamMap)
Revenues       = Revenues.groupby(['ds', 't'])[['y', 'Co', 'Cn', 'a', 'h']].sum().reset_index()
RevenuesByTeam = Revenues.groupby(['t'])['y'].sum().reset_index()
TeamsToRemove  = RevenuesByTeam.loc[RevenuesByTeam['y'] == 0, 't'].tolist()
Revenues       = Revenues[~Revenues['t'].isin(TeamsToRemove)]
Revenues       = Revenues[Revenues['t'] != 'Other']

del TeamMap, RevenuesByTeam, TeamsToRemove

In [ ]:
# Dataframe Setting & Cleaning - MacroEconomics & Merge
MacroEconomics = pd.DataFrame({
    'QuarterUpdateDate': [datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30), datetime(2024,6,30)],
    'Date': [datetime(2016,1,1), datetime(2016,2,1), datetime(2016,3,1), datetime(2016,4,1), datetime(2016,5,1), datetime(2016,6,1), datetime(2016,7,1), datetime(2016,8,1), datetime(2016,9,1), datetime(2016,10,1), datetime(2016,11,1), datetime(2016,12,1), datetime(2017,1,1), datetime(2017,2,1), datetime(2017,3,1), datetime(2017,4,1), datetime(2017,5,1), datetime(2017,6,1), datetime(2017,7,1), datetime(2017,8,1), datetime(2017,9,1), datetime(2017,10,1), datetime(2017,11,1), datetime(2017,12,1), datetime(2018,1,1), datetime(2018,2,1), datetime(2018,3,1), datetime(2018,4,1), datetime(2018,5,1), datetime(2018,6,1), datetime(2018,7,1), datetime(2018,8,1), datetime(2018,9,1), datetime(2018,10,1), datetime(2018,11,1), datetime(2018,12,1), datetime(2019,1,1), datetime(2019,2,1), datetime(2019,3,1), datetime(2019,4,1), datetime(2019,5,1), datetime(2019,6,1), datetime(2019,7,1), datetime(2019,8,1), datetime(2019,9,1), datetime(2019,10,1), datetime(2019,11,1), datetime(2019,12,1), datetime(2020,1,1), datetime(2020,2,1), datetime(2020,3,1), datetime(2020,4,1), datetime(2020,5,1), datetime(2020,6,1), datetime(2020,7,1), datetime(2020,8,1), datetime(2020,9,1), datetime(2020,10,1), datetime(2020,11,1), datetime(2020,12,1), datetime(2021,1,1), datetime(2021,2,1), datetime(2021,3,1), datetime(2021,4,1), datetime(2021,5,1), datetime(2021,6,1), datetime(2021,7,1), datetime(2021,8,1), datetime(2021,9,1), datetime(2021,10,1), datetime(2021,11,1), datetime(2021,12,1), datetime(2022,1,1), datetime(2022,2,1), datetime(2022,3,1), datetime(2022,4,1), datetime(2022,5,1), datetime(2022,6,1), datetime(2022,7,1), datetime(2022,8,1), datetime(2022,9,1), datetime(2022,10,1), datetime(2022,11,1), datetime(2022,12,1), datetime(2023,1,1), datetime(2023,2,1), datetime(2023,3,1), datetime(2023,4,1), datetime(2023,5,1), datetime(2023,6,1), datetime(2023,7,1), datetime(2023,8,1), datetime(2023,9,1), datetime(2023,10,1), datetime(2023,11,1), datetime(2023,12,1), datetime(2024,1,1), datetime(2024,2,1), datetime(2024,3,1), datetime(2024,4,1), datetime(2024,5,1), datetime(2024,6,1), datetime(2024,7,1), datetime(2024,8,1), datetime(2024,9,1), datetime(2024,10,1)],
    'Inflation': [0.003, -0.003, -0.002, -0.005, -0.003, -0.004, -0.001, -0.001, 0.001, -0.002, 0.001, 0.005, 0.010, 0.016, 0.014, 0.019, 0.014, 0.012, 0.011, 0.012, 0.011, 0.010, 0.009, 0.009, 0.009, 0.005, 0.008, 0.005, 0.010, 0.013, 0.015, 0.016, 0.014, 0.016, 0.016, 0.011, 0.009, 0.010, 0.010, 0.011, 0.008, 0.007, 0.004, 0.004, 0.003, 0.002, 0.002, 0.005, 0.005, 0.003, 0.001, 0.000, -0.002, -0.002, -0.004, -0.005, -0.006, -0.003, -0.002, -0.002, 0.004, 0.005, 0.008, 0.011, 0.013, 0.013, 0.020, 0.020, 0.025, 0.030, 0.037, 0.039, 0.048, 0.057, 0.065, 0.060, 0.068, 0.080, 0.079, 0.084, 0.089, 0.118, 0.118, 0.116, 0.100, 0.092, 0.076, 0.082, 0.076, 0.064, 0.059, 0.054, 0.053, 0.017, 0.007, 0.006, 0.008, 0.008, 0.012, 0.008, 0.008, 0.008, 0.013, 0.011, 0.007, 0.009], 
    'PILGrowth': [0.0012, 0.0012, 0.0012, 0.0007, 0.0007, 0.0007, 0.0018, 0.0018, 0.0018, 0.0012, 0.0012, 0.0012, 0.0016, 0.0016, 0.0016, 0.0015, 0.0015, 0.0015, 0.0013, 0.0013, 0.0013, 0.0018, 0.0018, 0.0018, -0.0003, -0.0003, -0.0003, 0.0002, 0.0002, 0.0002, 0.0003, 0.0003, 0.0003, 0.0010, 0.0010, 0.0010, 0.0003, 0.0003, 0.0003, 0.0011, 0.0011, 0.0011, 0.0002, 0.0002, 0.0002, -0.0022, -0.0022, -0.0022, -0.0200, -0.0200, -0.0200, -0.0393, -0.0393, -0.0393, 0.0463, 0.0463, 0.0463, -0.0017, -0.0017, -0.0017, 0.0050, 0.0050, 0.0050, 0.0087, 0.0087, 0.0087, 0.0097, 0.0097, 0.0097, 0.0027, 0.0027, 0.0027, 0.0007, 0.0007, 0.0007, 0.0047, 0.0047, 0.0047, 0.0013, 0.0013, 0.0013, -0.0003, -0.0003, -0.0003, 0.0013, 0.0013, 0.0013, -0.0007, -0.0007, -0.0007, 0.0010, 0.0010, 0.0010, 0.0003, 0.0003, 0.0003, 0.0010, 0.0010, 0.0010, 0.0007, 0.0007, 0.0007, 0.0000, 0.0000, 0.0000, 0.0000], 
    'UnEmployment': [0.073, 0.073, 0.073, 0.072, 0.072, 0.072, 0.066, 0.066, 0.066, 0.073, 0.073, 0.073, 0.072, 0.072, 0.072, 0.069, 0.069, 0.069, 0.065, 0.065, 0.065, 0.068, 0.068, 0.068, 0.070, 0.070, 0.070, 0.067, 0.067, 0.067, 0.058, 0.058, 0.058, 0.063, 0.063, 0.063, 0.063, 0.063, 0.063, 0.058, 0.058, 0.058, 0.055, 0.055, 0.055, 0.058, 0.058, 0.058, 0.053, 0.053, 0.053, 0.039, 0.039, 0.039, 0.055, 0.055, 0.055, 0.055, 0.055, 0.055, 0.055, 0.055, 0.055, 0.056, 0.056, 0.056, 0.052, 0.052, 0.052, 0.054, 0.054, 0.054, 0.052, 0.052, 0.052, 0.048, 0.048, 0.048, 0.043, 0.043, 0.043, 0.043, 0.043, 0.043, 0.045, 0.045, 0.042, 0.042, 0.042, 0.040, 0.040, 0.040, 0.040, 0.040, 0.040, 0.041, 0.041, 0.041, 0.042, 0.042, 0.042, 0.036, 0.036, 0.036, 0.036, 0.036], 
    'Interest': [0.0005, 0.0005, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0050, 0.0050, 0.0125, 0.0200, 0.0200, 0.0250, 0.0250, 0.0300, 0.0350, 0.0375, 0.0400, 0.0425, 0.0425, 0.0450, 0.0450, 0.0450, 0.0450, 0.0450, 0.0450, 0.0450, 0.0450, 0.0450, 0.0450, 0.0450, 0.0425, 0.0425, 0.0425, 0.0365, 0.0340], 
    'PublicExpense': [28733333333, 28733333333, 28733333333, 28766666667, 28766666667, 28766666667, 28766666667, 28766666667, 28766666667, 28933333333, 28933333333, 28933333333, 28833333333, 28833333333, 28833333333, 28866666667, 28866666667, 28866666667, 28866666667, 28866666667, 28866666667, 28966666667, 28966666667, 28966666667, 28966666667, 28966666667, 28966666667, 28933333333, 28933333333, 28933333333, 28833333333, 28833333333, 28833333333, 28800000000, 28800000000, 28800000000, 28800000000, 28800000000, 28800000000, 28766666667, 28766666667, 28766666667, 28766666667, 28766666667, 28766666667, 28666666667, 28666666667, 28666666667, 29033333333, 29033333333, 29033333333, 28466666667, 28466666667, 28466666667, 28933333333, 28933333333, 28933333333, 28933333333, 28933333333, 28933333333, 29433333333, 29433333333, 29433333333, 29433333333, 29433333333, 29433333333, 29533333333, 29533333333, 29533333333, 29666666667, 29666666667, 29666666667, 29633333333, 29633333333, 29633333333, 29700000000, 29700000000, 29700000000, 29600000000, 29600000000, 29600000000, 29900000000, 29900000000, 29900000000, 30066666667, 30066666667, 30066666667, 30233333333, 30233333333, 30233333333, 30400000000, 30400000000, 30400000000, 30433333333, 30433333333, 30433333333, 30200000000, 30200000000, 30200000000, 30500000000, 30500000000, 30500000000, 30500000000, 30500000000, 30500000000, 30500000000], 
    'StockMarket': [1.0000, -0.0554, 0.0280, 0.0267, -0.0309, -0.1014, 0.0401, 0.0057, -0.0320, 0.0442, -0.0114, 0.1361, -0.0335, 0.0174, 0.0835, 0.0057, 0.0059, -0.0071, 0.0438, 0.0085, 0.0474, 0.0043, -0.0187, -0.0230, 0.0757, -0.0383, -0.0087, -0.0700, -0.0826, -0.0169, 0.0273, -0.0876, 0.0218, -0.0802, 0.0073, -0.0451, 0.0768, 0.0471, 0.0303, 0.0280, -0.0950, 0.0724, 0.0077, -0.0035, 0.0368, 0.0265, 0.0249, 0.0106, -0.0115, -0.0539, -0.2244, 0.0375, 0.0287, 0.0647, -0.0146, 0.0284, -0.0315, -0.0564, 0.2295, 0.0078, -0.0297, 0.0592, 0.0788, -0.0206, 0.0426, -0.0027, 0.0104, 0.0255, -0.0125, 0.0464, -0.0395, 0.0594, -0.0195, -0.0521, -0.0155, -0.0307, 0.0104, -0.1310, 0.0522, -0.0378, -0.0422, 0.0970, 0.0864, -0.0367, 0.1220, 0.0330, -0.0014, -0.0379, 0.0837, 0.0501, -0.0274, -0.0204, -0.0177, 0.0719, 0.0207, 0.0129, 0.0597, 0.0666, -0.0289, 0.0221, -0.0388, 0.0184, 0.0180, -0.0072, 0.0046, 0.0115]
})

MacroEconomics      = MacroEconomics.rename(columns={'Date': 'ds'})
Revenues            = pd.merge(Revenues, MacroEconomics, on='ds')
Revenues            = Revenues.drop(columns=['QuarterUpdateDate'])
NumericColumns      = ['y', 'Co', 'Cn', 'a', 'h', 'Inflation', 'PILGrowth', 'UnEmployment', 'Interest', 'PublicExpense', 'StockMarket']
WinsorizationLimits = (0.05, 0.075)

TeamDataframes = {}
for team in Revenues['t'].unique():
    teamDf = Revenues[Revenues['t'] == team].drop(columns=['t']).reset_index(drop=True)
    for col in NumericColumns:
        teamDf[col].fillna(0, inplace=True)
        teamDf[col] = winsorize(teamDf[col], limits=WinsorizationLimits, inplace=True)
        teamDf[col] = teamDf[col].astype(float)
    TeamDataframes[team] = teamDf
    globals()[f"Revenues_{team}"] = teamDf

del MacroEconomics, NumericColumns, WinsorizationLimits, team, teamDf, col

In [ ]:
# Model, Fit & Prediction - Prophet
ItalyHolidays     = holidays.Italy(years=range(2013, 2035))
HolidaysList      = [{'ds': date, 'holiday': name} for date, name in ItalyHolidays.items()]
HolidaysDataframe = pd.DataFrame(HolidaysList)

AllForecasts = []
   
def FitAndForecast(df, teamName):
    if df.dropna().shape[0] < 2:
        print(f"Skipping forecast for team {teamName} due to insufficient data.")
        return
    
    Cap   = max(df['y'].max(), 2)
    Floor = df['y'].min()

    Model = Prophet(
        growth='logistic', seasonality_mode='multiplicative',
        yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True,
        interval_width=0.7, changepoint_range=1, changepoint_prior_scale=0.1, seasonality_prior_scale=5, holidays_prior_scale=5,
        holidays=HolidaysDataframe)
    Model.add_seasonality(name='quarterly', period=365.25/4, fourier_order=5)
    Model.add_seasonality(name='monthly', period=365.25/12, fourier_order=3)

    df['cap']   = Cap
    df['floor'] = Floor
    Model.fit(df)
    
    Future           = Model.make_future_dataframe(periods=24, freq='MS')
    Future['cap']    = Cap
    Future['floor']  = Floor
    Forecast         = Model.predict(Future)
    Forecast['team'] = teamName
    
    AllForecasts.append(Forecast)

with tqdm(total=len(TeamDataframes), desc="Processing forecasts", unit="team") as pbar:
    for teamName, teamDf in TeamDataframes.items():
        pbar.set_description(f"Processing forecasts for {teamName}")
        FitAndForecast(teamDf, teamName)
        pbar.update(1)

Forecasts = pd.concat(AllForecasts, ignore_index=True)
Forecasts = Forecasts[['ds', 'team', 'yhat', 'yhat_lower', 'yhat_upper']]

Revenues['ds_team']  = Revenues['ds'].astype(str) + '_' + Revenues['t']
Forecasts['ds_team'] = Forecasts['ds'].astype(str) + '_' + Forecasts['team']
Forecasts            = Forecasts.merge(Revenues[['ds_team', 'y', 'Co', 'Cn', 'a', 'h', 'Inflation', 'PILGrowth', 'UnEmployment', 'Interest', 'PublicExpense', 'StockMarket']], on='ds_team', how='left')
Forecasts            = Forecasts.rename(columns={'y': 'y_actual'})
Forecasts.drop(columns=['ds_team'], inplace=True)
Revenues.drop(columns=['ds_team'], inplace=True)

Forecasts              = Forecasts[(Forecasts['y_actual'] != 0) | (Forecasts['y_actual'].isna())]
Forecasts['residuals'] = Forecasts['y_actual'] - Forecasts['yhat']

Forecasts['AEProphet']  = (Forecasts['residuals']).abs()
Forecasts['APEProphet'] = (Forecasts['residuals']) / Forecasts['y_actual']

del HolidaysDataframe, HolidaysList, ItalyHolidays, pbar, teamDf, teamName, AllForecasts, TeamDataframes
[globals().pop(var) for var in list(globals()) if var.startswith("Revenues_")]

Processing forecasts for Commercial:   0%|          | 0/19 [00:00<?, ?team/s]09:49:50 - cmdstanpy - INFO - Chain [1] start processing
09:49:51 - cmdstanpy - INFO - Chain [1] done processing
Processing forecasts for Corporate:   5%|▌         | 1/19 [00:01<00:20,  1.13s/team] 09:49:52 - cmdstanpy - INFO - Chain [1] start processing
09:49:52 - cmdstanpy - INFO - Chain [1] done processing
Processing forecasts for DueDiligence:  11%|█         | 2/19 [00:02<00:17,  1.04s/team]09:49:52 - cmdstanpy - INFO - Chain [1] start processing
09:49:53 - cmdstanpy - INFO - Chain [1] done processing
Processing forecasts for Energy:  16%|█▌        | 3/19 [00:03<00:18,  1.13s/team]      09:49:54 - cmdstanpy - INFO - Chain [1] start processing
09:49:54 - cmdstanpy - INFO - Chain [1] done processing
Processing forecasts for Engineering:  21%|██        | 4/19 [00:04<00:16,  1.09s/team]09:49:55 - cmdstanpy - INFO - Chain [1] start processing
09:49:55 - cmdstanpy - INFO - Chain [1] done processing
Processing fo

[            ds              y             Co    Cn      a      h  Inflation  \
 0   2016-01-01       0.000000   30770.140000   9.0   11.0   24.0      0.003   
 1   2016-02-01       0.000000   66179.140000   9.0   11.0    9.0     -0.003   
 2   2016-03-01       0.000000   78128.020000   9.0   11.0   32.0     -0.002   
 3   2016-04-01       0.000000   61169.560000   9.0   12.0   31.0     -0.003   
 4   2016-05-01       0.000000   42694.360000   9.0   11.0   31.0     -0.003   
 ..         ...            ...            ...   ...    ...    ...        ...   
 101 2024-06-01  387755.411220  180488.241925  72.0  152.0   63.0      0.008   
 102 2024-07-01  321289.220155  164731.081440  70.0  125.0  105.0      0.013   
 103 2024-08-01  324075.695980  113914.006115  42.0   92.0  105.0      0.011   
 104 2024-09-01  351973.339940  137403.507990  57.0  113.0  104.0      0.007   
 105 2024-10-01  135152.035390  132118.928430  66.0  121.0  106.0      0.009   
 
      PILGrowth  UnEmployment  Interes

In [ ]:
# Model, Fit & Prediction - Ensemble Regression 
ColumnsToFill = ['Co', 'Cn', 'a', 'h', 'Inflation', 'PILGrowth', 'UnEmployment', 'Interest', 'PublicExpense', 'StockMarket']
Forecasts     = Forecasts.sort_values(['team', 'ds'])
for team, group in Forecasts.groupby('team'):
    for column in ColumnsToFill: Forecasts.loc[group.index, column] = group[column].fillna(group[column].rolling(window=24, min_periods=1).mean())

RegressionResults     = []
RegressionPredictions = []

Forecasts['year']  = pd.to_datetime(Forecasts['ds']).dt.year
Forecasts['month'] = pd.to_datetime(Forecasts['ds']).dt.month

for team in Forecasts['team'].unique():
    teamData    = Forecasts[(Forecasts['team'] == team) & (Forecasts['y_actual'].notna())]
    newTeamData = Forecasts[(Forecasts['team'] == team) & (Forecasts['y_actual'].isna())]

    if len(teamData) < 2: continue
    
    x    = teamData[['Co', 'Cn', 'a', 'h', 'year', 'month', 'Inflation', 'PILGrowth', 'UnEmployment', 'Interest', 'PublicExpense', 'StockMarket']]
    y    = teamData['y_actual']
    xNew = newTeamData[['Co', 'Cn', 'a', 'h', 'year', 'month', 'Inflation', 'PILGrowth', 'UnEmployment', 'Interest', 'PublicExpense', 'StockMarket']]
    xNew = xNew[xNew['year'].notna() & xNew['Cn'].notna()]
    yNew = newTeamData['y_actual']
    
    xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.2, random_state=42)
    
    numericFeatures = x.columns
    preprocessor    = ColumnTransformer(transformers=[('num', StandardScaler(), numericFeatures),])
    
    ridgeModel = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=5)),
        ('poly', PolynomialFeatures(degree=2, include_bias=True)),
        ('ridge', RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5))])

    lassoModel = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=5)),
        ('poly', PolynomialFeatures(degree=2, include_bias=True)),
        ('lasso', LassoCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5, max_iter=10000, tol=1e-2))])

    ridgeModel.fit(xTrain, yTrain)
    lassoModel.fit(xTrain, yTrain)
    yTrainPredRidge = ridgeModel.predict(xTrain)
    yTestPredRidge  = ridgeModel.predict(xTest)
    yTrainPredLasso = lassoModel.predict(xTrain)
    yTestPredLasso  = lassoModel.predict(xTest)
    yNewPredLasso   = lassoModel.predict(xNew)
    yNewPredRidge   = ridgeModel.predict(xNew)

    MAETrainRidge = mean_absolute_error(yTrain, yTrainPredRidge)
    MAETrainLasso = mean_absolute_error(yTrain, yTrainPredLasso)
    MAETestRidge  = mean_absolute_error(yTest, yTestPredRidge)
    MAETestLasso  = mean_absolute_error(yTest, yTestPredLasso)
    
    WeightRidgeTrain = 1 / MAETrainRidge
    WeightLassoTrain = 1 / MAETrainLasso
    WeightRidgeTest  = 1 / MAETestRidge
    WeightLassoTest  = 1 / MAETestLasso
    TotalWeightTrain = WeightRidgeTrain + WeightLassoTrain
    TotalWeightTest  = WeightRidgeTest + WeightLassoTest
    WeightRidgeTrain /= TotalWeightTrain
    WeightLassoTrain /= TotalWeightTrain
    WeightRidgeTest  /= TotalWeightTest
    WeightLassoTest  /= TotalWeightTest

    yTrainPredCombined = (yTrainPredRidge * WeightRidgeTrain + yTrainPredLasso * WeightLassoTrain)
    yTestPredCombined  = (yTestPredRidge * WeightRidgeTest + yTestPredLasso * WeightLassoTest)
    yNewPredCombined   = (yNewPredRidge * WeightRidgeTest + yNewPredLasso * WeightLassoTest)

    MAETrainCombined  = mean_absolute_error(yTrain, yTrainPredCombined)
    MAPETrainCombined = round(mean_absolute_percentage_error(yTrain, yTrainPredCombined) * 100)
    R2TrainCombined   = round(r2_score(yTrain, yTrainPredCombined) * 100)
    MAETestCombined   = mean_absolute_error(yTest, yTestPredCombined)
    MAPETestCombined  = round(mean_absolute_percentage_error(yTest, yTestPredCombined) * 100)
    R2TestCombined    = round(r2_score(yTest, yTestPredCombined) * 100)
    TrainTestDistance = round((MAPETestCombined - MAPETrainCombined) * 100)

    PredictionTrainCombined = pd.DataFrame({'Team': team, 'Set': 'Train', 'y_actual': yTrain, 'y_pred': yTrainPredCombined })
    PredictionTestCombined  = pd.DataFrame({'Team': team, 'Set': 'Test', 'y_actual': yTest, 'y_pred': yTestPredCombined})
    PredictionNewCombined   = pd.DataFrame({'Team': team, 'Set': 'New', 'y_actual': None, 'y_pred': yNewPredCombined})

    RegressionResults.append({'Team': team, 'MAPE Train': MAPETrainCombined, 'MAE Test': MAETestCombined, 'MAPE Test (%)': f"{MAPETestCombined}%", 'R2 Test': f"{R2TestCombined}%", "TrainTestDistance": f"{TrainTestDistance}%"})
    RegressionPredictions.append(PredictionTrainCombined)
    RegressionPredictions.append(PredictionTestCombined)
    RegressionPredictions.append(PredictionNewCombined)
    PredictionTrainCombined['ds'] = pd.to_datetime(dict(year=xTrain['year'], month=xTrain['month'], day=[1]*len(xTrain)))
    PredictionTestCombined['ds']  = pd.to_datetime(dict(year=xTest['year'], month=xTest['month'], day=[1]*len(xTest)))
    PredictionNewCombined['ds']   = pd.to_datetime(dict(year=newTeamData['year'].values, month=newTeamData['month'].values, day=[1]*len(newTeamData)))
    RegressionPredictions.extend([PredictionTrainCombined, PredictionTestCombined, PredictionNewCombined])

RegressionResults     = pd.DataFrame(RegressionResults)
RegressionPredictions = pd.concat(RegressionPredictions).reset_index(drop=True)

del team, group, column, ColumnsToFill, teamData, newTeamData
del lassoModel, preprocessor, numericFeatures, ridgeModel, x, xTest, xTrain, y, yTest, yTrain, yTestPredCombined, yTestPredLasso, yTestPredRidge, yTrainPredCombined, yTrainPredLasso, yTrainPredRidge
del MAETestCombined, MAETestLasso, MAETestRidge, MAETrainCombined, MAETrainLasso, MAETrainRidge, MAPETestCombined, MAPETrainCombined, PredictionTestCombined, PredictionTrainCombined
del R2TestCombined, R2TrainCombined, TotalWeightTest, TotalWeightTrain, TrainTestDistance, WeightLassoTest, WeightLassoTrain, WeightRidgeTest, WeightRidgeTrain
del xNew, yNew, yNewPredLasso, yNewPredRidge, yNewPredCombined, PredictionNewCombined

In [ ]:
# Combining Results
RegressionPredictions            = RegressionPredictions[RegressionPredictions['Set'] != 'Train'].copy()
RegressionPredictions['ds_team'] = RegressionPredictions['ds'].astype(str) + '_' + RegressionPredictions['Team']

Forecasts['ds_team'] = Forecasts['ds'].astype(str) + '_' + Forecasts['team']
Forecasts            = Forecasts.merge(RegressionPredictions[['ds_team', 'y_pred']], on='ds_team', how='left')
Forecasts.rename(columns={'y_pred': 'RegressionForecast'}, inplace=True)
Forecasts.drop(columns=['ds_team'], inplace=True)

RegressionResults.rename(columns={'Team': 'team'}, inplace=True)
Forecasts = Forecasts.merge(RegressionResults[['team', 'MAE Test', 'MAPE Test (%)', 'R2 Test', 'TrainTestDistance']], on='team', how='left')
Forecasts.rename(columns={'MAE Test': 'MAE Regression', 'MAPE Test (%)': 'MAPE Regression', 'R2 Test': 'R2 Regression', 'TrainTestDistance': 'TrainTestDistanceRegression'}, inplace=True)

Forecasts['MAPE Regression'] = Forecasts['MAPE Regression'].apply(lambda x: float(x.strip('%')) / 100 if pd.notna(x) else None)

def CalculateConditionalEnsemble(row, prophetCol, regressionCol, mapeCol):
    if regressionCol is None: return prophetCol
    elif -0.25 <= row[mapeCol] <= 0.25: return (row[prophetCol] + row[regressionCol]) / 2
    else: return row[prophetCol]

Forecasts['FinalEnsembleForecast'] = Forecasts.apply(lambda row: CalculateConditionalEnsemble(row, 'yhat', 'RegressionForecast', 'MAPE Regression'), axis=1)
Forecasts['FinalEnsembleAE']       = Forecasts.apply(lambda row: CalculateConditionalEnsemble(row, 'AEProphet', 'MAE Regression', 'MAPE Regression'), axis=1)
Forecasts['FinalEnsembleAPE']      = Forecasts.apply(lambda row: CalculateConditionalEnsemble(row, 'APEProphet', 'MAPE Regression', 'MAPE Regression'), axis=1)

Forecasts['FinalEnsembleForecast'] = Forecasts['FinalEnsembleForecast'].fillna(Forecasts['yhat'])
Forecasts['FinalEnsembleAE']       = Forecasts['FinalEnsembleAE'].fillna(Forecasts['AEProphet'])
Forecasts['FinalEnsembleAPE']      = Forecasts['FinalEnsembleAPE'].fillna(Forecasts['APEProphet'])

Forecasts['FinalEnsembleForecast'] = Forecasts['FinalEnsembleForecast'].round(0).astype(int)
Forecasts['FinalEnsembleForecast'] = Forecasts['FinalEnsembleForecast'].apply(lambda x: max(x, 0))
Forecasts['FinalEnsembleAE']       = Forecasts['FinalEnsembleAE'].round(0)
Forecasts['FinalEnsembleAPE']      = Forecasts['FinalEnsembleAPE'].round(2)
Forecasts['FinalEnsembleAE']       = Forecasts.apply(lambda row: row['FinalEnsembleAE'] if pd.notna(row['y_actual']) else None, axis=1)
Forecasts['FinalEnsembleAPE']      = Forecasts.apply(lambda row: row['FinalEnsembleAPE'] if pd.notna(row['y_actual']) else None, axis=1)

FinalForecasts = Forecasts[['ds', 'team', 'yhat_lower', 'yhat_upper', 'y_actual', 'FinalEnsembleForecast', 'FinalEnsembleAE', 'FinalEnsembleAPE']]

del RegressionPredictions, RegressionResults, Forecasts

In [ ]:
# Plotting & Diagnostics
def PlotHistoForecast(df, selectedTeams):
    Today = datetime.now()
    FirstDayCurrentMonth = Today.replace(day=1)
    LastDayPreviousMonth = FirstDayCurrentMonth - timedelta(days=1)

    fig, axs = plt.subplots(3, 1, figsize=(12, 16), gridspec_kw={'height_ratios': [1, 1, 1]})
    
    selectedDf = df[df['team'].isin(selectedTeams)]
    pastData   = selectedDf[selectedDf['ds'] <= LastDayPreviousMonth]
    futureData = selectedDf[selectedDf['ds'] > LastDayPreviousMonth]

    axs[0].plot(pastData['ds'], pastData['y_actual'], color='#2C3E50', alpha=0.8, marker='o', label='Actual', linewidth=2, solid_capstyle='round', solid_joinstyle='round', markersize=4)
    axs[0].plot(pastData['ds'], pastData['FinalEnsembleForecast'], color='#E67E22', alpha=0.6, marker='o', label='Past Estimate', linewidth=1.5, linestyle='--', solid_capstyle='round', solid_joinstyle='round', markersize=4)
    axs[0].plot(futureData['ds'], futureData['FinalEnsembleForecast'], color='#E67E22', alpha=0.6, marker='o', label='Forecast', linewidth=1.5, linestyle='--', solid_capstyle='round', solid_joinstyle='round', markersize=4)
    axs[0].fill_between(futureData['ds'], futureData['yhat_lower'], futureData['yhat_upper'], color='#E67E22', alpha=0.2, joinstyle='round', hatch='//')

    axs[0].set_title('Revenue Forecast with Confidence Intervals', fontsize=13, fontweight='bold', pad=15)
    axs[0].set_xlabel('Date', fontsize=10)
    axs[0].set_ylabel('Revenue (€)', fontsize=10)
    axs[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axs[0].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    axs[0].tick_params(axis='x', rotation=45)
    axs[0].yaxis.set_major_locator(MaxNLocator(nbins=10))

    handles, labels = axs[0].get_legend_handles_labels()
    byLabel         = dict(zip(labels, handles))
    legend          = axs[0].legend(byLabel.values(), byLabel.keys(), loc='upper left', frameon=False, fontsize=9, ncol=3, borderpad=1, handlelength=0.5, handletextpad=0.5)

    axs[0].grid(visible=True, which='major', color='#B0BEC5', linestyle='--', linewidth=0.8, alpha=0.6, solid_capstyle='round', solid_joinstyle='round')
    axs[0].grid(visible=True, which='minor', color='#CFD8DC', linestyle='--', linewidth=0.4, alpha=0.4, solid_capstyle='round', solid_joinstyle='round')
    axs[0].minorticks_on()

    # Absolute Error (AE) Plot
    axs[1].plot(pastData['ds'], pastData['FinalEnsembleAE'], color='#16A085', marker='o', linewidth=1.5, alpha=0.33, label='AE', solid_capstyle='round', solid_joinstyle='round', markersize=4, linestyle='--')
    MAE = pastData['FinalEnsembleAE'].mean()
    axs[1].axhline(y=MAE, color='#16A085', linestyle='-', linewidth=2.5, label=f'MAE ({MAE:.2f})')

    axs[1].set_title('Absolute Error (AE) Over Time', fontsize=13, fontweight='bold', pad=15)
    axs[1].set_xlabel('Date', fontsize=10)
    axs[1].set_ylabel('AE')
    axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axs[1].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    axs[1].tick_params(axis='x', rotation=45)
    axs[1].yaxis.set_major_locator(MaxNLocator(nbins=10))

    axs[1].legend(loc='upper left', frameon=False, fontsize=9, borderpad=1, handlelength=1, handletextpad=0.5, ncols=2)
    axs[1].grid(visible=True, color='#B0BEC5', linestyle='--', linewidth=0.8, alpha=0.6, solid_capstyle='round', solid_joinstyle='round')
    axs[1].minorticks_on()

    # Absolute Percentage Error (APE) Plot
    axs[2].plot(pastData['ds'], pastData['FinalEnsembleAPE'], color='#2980B9', marker='o', linewidth=1.5, alpha=0.33, label='APE', solid_capstyle='round', solid_joinstyle='round', markersize=4, linestyle='--')
    MAPE = pastData['FinalEnsembleAPE'].mean()
    axs[2].axhline(y=MAPE, color='#2980B9', linestyle='-', linewidth=2.5, label=f'MAPE ({100*MAPE:.2f}%)')

    axs[2].set_title('Absolute Percentage Error (APE) Over Time', fontsize=13, fontweight='bold', pad=15)
    axs[2].set_xlabel('Date', fontsize=10)
    axs[2].set_ylabel('APE')
    axs[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axs[2].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    axs[2].tick_params(axis='x', rotation=45)
    axs[2].yaxis.set_major_locator(MaxNLocator(nbins=10))

    axs[2].legend(loc='upper left', frameon=False, fontsize=9, borderpad=1, handlelength=1, handletextpad=0.5, ncols=2)
    axs[2].grid(visible=True, color='#B0BEC5', linestyle='--', linewidth=0.8, alpha=0.6, solid_capstyle='round', solid_joinstyle='round')
    axs[2].minorticks_on()

    plt.tight_layout()
    plt.show()

AllTeams     = sorted(FinalForecasts['team'].dropna().astype(str).unique())
TeamSelector = widgets.SelectMultiple(options=AllTeams, value=AllTeams, description='Teams', disabled=False, layout=widgets.Layout(height='365px', width='300px', padding='5px'))
Output       = widgets.interactive_output(PlotHistoForecast, {'df': widgets.fixed(FinalForecasts), 'selectedTeams': TeamSelector})
UI           = VBox([TeamSelector, Output], layout=Layout(align_items='initial', padding='20px', width='100%'))
display(UI)

del AllTeams, GridspecLayout, Layout, Output, TeamSelector, UI, VBox

In [ ]:
# Synthetic Diagnostics
from ipywidgets import widgets, Layout, VBox, HBox

Today                = pd.to_datetime("today")
LastDayPreviousMonth = pd.to_datetime(f"{Today.year}-{Today.month}-01") - pd.DateOffset(days=1)
FilteredData         = FinalForecasts[FinalForecasts['ds'] <= LastDayPreviousMonth]

AllTeams             = sorted(FilteredData['team'].unique())
TeamSelector         = widgets.SelectMultiple(options=AllTeams, value=AllTeams, description='Teams', disabled=False, layout=widgets.Layout(height='365px', width='300px', padding='5px'))
DateRange            = widgets.SelectionRangeSlider(
    options=pd.date_range(start=FilteredData['ds'].min(), end=FilteredData['ds'].max(), freq='M'),
    index=(0, len(pd.date_range(start=FilteredData['ds'].min(), end=FilteredData['ds'].max(), freq='M')) - 1),
    description='Date Range', layout=Layout(width='1000'), style={'description_width': 'initial'})

def PlotCumulativeMAPE(df, selectedTeams, dateRange):
    dfFiltered = df[(df['team'].isin(selectedTeams)) & (df['ds'] >= dateRange[0]) & (df['ds'] <= dateRange[1])].sort_values(by='ds')
    plt.figure(figsize=(12, 6))
    
    for team in selectedTeams:
        teamData       = dfFiltered[dfFiltered['team'] == team].set_index('ds').sort_index()
        cumulativeMAPE = teamData['FinalEnsembleAPE'].expanding(min_periods=1).mean() * 100
        plt.plot(cumulativeMAPE.index, cumulativeMAPE, label=f'{team} Cumulative APE (%)', color="#1c1c1c", linestyle='-', marker='o', linewidth=1.5, alpha=0.6)

    plt.title("Cumulative MAPE Over Time", fontsize=13, fontweight='bold', pad=15)
    plt.xlabel("Date", fontsize=10)
    plt.ylabel("APE", fontsize=10)
    
    plt.grid(visible=True, which='major', color='#B0BEC5', linestyle='--', linewidth=0.8, alpha=0.6)
    plt.grid(visible=True, which='minor', color='#CFD8DC', linestyle='--', linewidth=0.4, alpha=0.4)
    plt.gca().minorticks_on()
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m"))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

Output     = widgets.interactive_output(PlotCumulativeMAPE, {'df': widgets.fixed(FilteredData), 'selectedTeams': TeamSelector, 'dateRange': DateRange})
FiltersBox = VBox([TeamSelector, DateRange], layout=Layout(align_items='flex-start', padding='10px', width='500'))
UI         = HBox([FiltersBox, Output], layout=Layout(align_items='center', padding='20px', width='100%'))
display(UI)

del FilteredData, Today, LastDayPreviousMonth, HBox
del AllTeams, DateRange, FiltersBox, Layout, Output, TeamSelector, UI, VBox